# StyleGAN3 / Autolume — Latent Space Viewer (Colab, single T4)

Upload a trained `.pkl` checkpoint and explore what it learned: generate images from seeds, interpolate smoothly between two points in latent space ("traverse"), adjust truncation, and extract PCA/GANSpace-style salient latent directions.

**Note on the GUI:** the real `visualizer.py` from the repo is a windowed OpenGL/imgui desktop app — it can't run inside Colab's browser environment without heavy VNC-streaming setup. This notebook instead spins up a lightweight **Gradio web UI** that does the same job (upload a pkl, generate, interpolate) natively in the notebook's output — no extra streaming setup needed.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
# Confirm the T4 is attached
!nvidia-smi

Wed Sep  2 15:37:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Clone the Autolume fork of StyleGAN3 (contains legacy.py / dnnlib / torch_utils needed to load the pkl)
!git clone -b autolume https://github.com/Metacreation-Lab/stylegan3.git
%cd stylegan3

Cloning into 'stylegan3'...
remote: Enumerating objects: 319, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 319 (delta 105), reused 100 (delta 100), pack-reused 137 (from 3)
Receiving objects: 100% (319/319), 4.36 MiB | 9.85 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/stylegan3


In [ ]:
# Install dependencies. Colab already ships a CUDA-enabled torch, so we leave torch alone.
!apt-get -qq update && apt-get -qq install -y ninja-build > /dev/null
!pip install -q click requests tqdm pillow scipy scikit-learn gradio kornia

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.9 MB/s eta 0:00:00


In [ ]:
import sys, os
sys.path.append(os.getcwd())

import torch
import numpy as np
import legacy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Load a model and explore it

Run the cell below, then in the panel that appears:
1. Upload your `.pkl` and click **Load Model**.
2. Use the **Explore (manual)** tab to step through images by hand: pick **Seed A** / **Seed B** (or hit 🎲 Randomize), then drag **Interpolate A → B** to move smoothly between them. Adjust **Truncation psi** — lower values (~0.5–0.7) give more "average"/stable results, higher values (~1.0+) give more variety and artifacts.
3. Use the **Live Walk (auto)** tab for continuous, hands-off traversal — click **Start Walk** and it keeps wandering through latent space on its own (closest browser equivalent to the native visualizer's live movement); click **Stop Walk** to freeze on the current frame.

The first generation after loading will be slower (~30–60s) while the custom CUDA ops compile. After that, each image should take well under a second on the T4.

In [ ]:
import gradio as gr
import PIL.Image
import random
import time
import numpy as np
import torch

from sklearn.decomposition import PCA

G = None

# GANSpace-style feature state.
# We use PCA over sampled W-space points from the loaded generator.  Each
# component is a salient direction in the model's latent space and can be
# applied to W to explore the corresponding visual change.
GANSPACE = {
    'components': None,
    'mean': None,
    'explained_variance_ratio': None,
    'n_samples': 0,
    'model_id': None,
}

def load_model(pkl_file):
    global G, GANSPACE
    if pkl_file is None:
        return 'Upload a .pkl file first.'
    pkl_path = pkl_file if isinstance(pkl_file, str) else pkl_file.name
    with open(pkl_path, 'rb') as f:
        G = legacy.load_network_pkl(f)['G_ema'].to(device)
    G.eval()
    GANSPACE = {
        'components': None,
        'mean': None,
        'explained_variance_ratio': None,
        'n_samples': 0,
        'model_id': pkl_path,
    }
    return f'Loaded model — resolution {G.img_resolution}x{G.img_resolution}, z_dim {G.z_dim}, c_dim {G.c_dim}'

def make_w(seed, psi):
    z = torch.from_numpy(np.random.RandomState(int(seed)).randn(1, G.z_dim)).to(device).float()
    label = torch.zeros([1, G.c_dim], device=device)
    return G.mapping(z, label, truncation_psi=psi)

def w_to_image(w):
    with torch.no_grad():
        img = G.synthesis(w, noise_mode='const')
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)[0].cpu().numpy()
    return PIL.Image.fromarray(img)

def generate(seed_a, seed_b, interp, psi):
    if G is None:
        return None
    w_a = make_w(seed_a, psi)
    w_b = make_w(seed_b, psi)
    w = w_a * (1 - interp) + w_b * interp
    return w_to_image(w)

def randomize():
    return random.randint(0, 999999), random.randint(0, 999999)

# --- Live walk: continuously wander the latent space, yielding frames ---
walking = {'active': False}

def latent_walk(psi, fps, steps_per_leg):
    if G is None:
        yield None
        return
    current_w = make_w(random.randint(0, 999999), psi)
    while walking['active']:
        target_w = make_w(random.randint(0, 999999), psi)
        for step in range(int(steps_per_leg)):
            if not walking['active']:
                return
            t = step / int(steps_per_leg)
            w = current_w * (1 - t) + target_w * t
            yield w_to_image(w)
            time.sleep(1 / fps)
        current_w = target_w

def start_walk(psi, fps, steps_per_leg):
    walking['active'] = True
    yield from latent_walk(psi, fps, steps_per_leg)

def stop_walk():
    walking['active'] = False

# -------------------------------------------------------------------------
# GANSpace-style feature extraction
# -------------------------------------------------------------------------

def extract_salient_features(num_features, num_samples, sparsity, seed):
    """
    Extract PCA directions from sampled W-space points.

    This follows the workshop's GANSpace idea: sample the model's latent
    space, use PCA to find directions explaining the largest variations,
    then expose those directions as editable controls.

    'sparsity' is implemented as a soft threshold on each PCA direction.
    At 1.0 the full PCA direction is retained. Lower values keep only the
    largest-magnitude coordinates, producing a sparser edit direction.
    """
    global GANSPACE

    if G is None:
        return 'Load a .pkl model first.', None, None, None

    num_features = int(num_features)
    num_samples = int(num_samples)
    sparsity = float(sparsity)

    if num_features < 1:
        return 'Number of features must be at least 1.', None, None, None
    if num_samples <= num_features:
        return f'Number of samples must be greater than number of features ({num_features}).', None, None, None

    try:
        # Sample W. For StyleGAN/StyleGAN3, G.mapping returns [B, num_ws, w_dim].
        # Averaging over style layers gives one W vector per sampled latent point.
        rng = np.random.RandomState(int(seed))
        z = torch.from_numpy(rng.randn(num_samples, G.z_dim)).to(device).float()
        labels = torch.zeros([num_samples, G.c_dim], device=device)

        with torch.no_grad():
            w = G.mapping(z, labels, truncation_psi=1.0)

        w_samples = w.mean(dim=1).detach().cpu().numpy().astype(np.float32)

        # PCA gives ordered directions: component 1 explains the most variance,
        # component N explains less, matching the workshop description.
        pca = PCA(n_components=num_features, svd_solver='randomized',
                  random_state=int(seed))
        pca.fit(w_samples)

        components = pca.components_.astype(np.float32)
        mean_w = pca.mean_.astype(np.float32)

        # Apply a simple sparsity control to the direction vectors.
        # sparsity=1 -> keep all coordinates.
        # Lower values -> retain only the strongest coordinates.
        if sparsity < 0.999:
            keep = max(1, int(round(components.shape[1] * sparsity)))
            sparse_components = np.zeros_like(components)
            for i, comp in enumerate(components):
                idx = np.argpartition(np.abs(comp), -keep)[-keep:]
                sparse_components[i, idx] = comp[idx]
                norm = np.linalg.norm(sparse_components[i])
                if norm > 0:
                    sparse_components[i] /= norm
            components = sparse_components

        GANSPACE = {
            'components': components,
            'mean': mean_w,
            'explained_variance_ratio': pca.explained_variance_ratio_.astype(np.float32),
            'n_samples': num_samples,
            'model_id': GANSPACE.get('model_id'),
        }

        rows = []
        for i, ratio in enumerate(GANSPACE['explained_variance_ratio'], start=1):
            rows.append(f'Feature {i:02d}: {ratio * 100:.2f}% variance')
        summary = (
            f'Extracted {num_features} salient directions from {num_samples} W samples.\n'
            f'PCA directions are ordered from highest to lower explained variance.\n'
            + '\n'.join(rows)
        )

        # Update the Dropdown's choices and selected value explicitly.
        # Returning a plain list here makes newer Gradio versions interpret
        # the list as the Dropdown *value*, which later arrives in callbacks
        # as a list and causes int(feature_index) to fail.
        feature_choices = list(range(1, num_features + 1))
        return summary, gr.update(choices=feature_choices, value=1), 0.0

    except Exception as e:
        return f'Feature extraction failed: {type(e).__name__}: {e}', None, None

def _feature_number(feature_index):
    # Gradio may supply a single selected Dropdown value as a list in some
    # versions/state transitions. Normalize it here.
    if isinstance(feature_index, (list, tuple, np.ndarray)):
        if len(feature_index) == 0:
            return None
        feature_index = feature_index[0]
    try:
        return int(feature_index)
    except (TypeError, ValueError):
        return None

def feature_info(feature_index):
    if GANSPACE['components'] is None:
        return 'Extract features first.'
    feature_number = _feature_number(feature_index)
    if feature_number is None:
        return 'Select a feature first.'
    i = feature_number - 1
    if i < 0 or i >= len(GANSPACE['components']):
        return 'Invalid feature.'
    ratio = float(GANSPACE['explained_variance_ratio'][i]) * 100
    norm = float(np.linalg.norm(GANSPACE['components'][i]))
    return f'Feature {i+1}: PCA direction {i+1} — explained variance {ratio:.2f}% — direction norm {norm:.3f}'

def apply_feature(seed, psi, feature_index, strength, layer_mode):
    if G is None:
        return None, None, 'Load a .pkl model first.'
    if GANSPACE['components'] is None:
        return None, None, 'Extract salient features first.'

    feature_number = _feature_number(feature_index)
    if feature_number is None:
        return None, None, 'Select a feature first.'
    i = feature_number - 1
    components = GANSPACE['components']
    if i < 0 or i >= len(components):
        return None, None, 'Invalid feature index.'

    # Use the same W point for the base and edited result.
    base_w = make_w(int(seed), float(psi))
    direction = torch.from_numpy(components[i]).to(device=device, dtype=base_w.dtype)

    # Broadcast the PCA direction across W's style layers.  This is the
    # standard W-space form of a global GANSpace edit.
    direction = direction.view(1, 1, -1)

    if layer_mode == 'All layers':
        edited_w = base_w + float(strength) * direction
    elif layer_mode == 'Coarse layers':
        n = base_w.shape[1]
        mask = torch.zeros((1, n, 1), device=device, dtype=base_w.dtype)
        mask[:, :max(1, n // 3), :] = 1
        edited_w = base_w + float(strength) * direction * mask
    elif layer_mode == 'Fine layers':
        n = base_w.shape[1]
        mask = torch.zeros((1, n, 1), device=device, dtype=base_w.dtype)
        mask[:, (2 * n) // 3:, :] = 1
        edited_w = base_w + float(strength) * direction * mask
    else:
        edited_w = base_w + float(strength) * direction

    base_img = w_to_image(base_w)
    edited_img = w_to_image(edited_w)

    return base_img, edited_img, feature_info(feature_index)

def save_features():
    if GANSPACE['components'] is None:
        return None
    path = '/content/ganspace_features.npz'
    np.savez(
        path,
        components=GANSPACE['components'],
        mean=GANSPACE['mean'],
        explained_variance_ratio=GANSPACE['explained_variance_ratio'],
        n_samples=GANSPACE['n_samples'],
    )
    return path

def load_features(feature_file):
    global GANSPACE
    if feature_file is None:
        return 'Choose a .npz feature file first.'
    try:
        feature_path = feature_file if isinstance(feature_file, str) else feature_file.name
        data = np.load(feature_path)
        GANSPACE['components'] = data['components'].astype(np.float32)
        GANSPACE['mean'] = data['mean'].astype(np.float32)
        GANSPACE['explained_variance_ratio'] = data['explained_variance_ratio'].astype(np.float32)
        GANSPACE['n_samples'] = int(data.get('n_samples', 0))
        choices = list(range(1, len(GANSPACE['components']) + 1))
        return f'Loaded {len(choices)} feature directions.', gr.update(choices=choices, value=1)
    except Exception as e:
        return f'Could not load feature file: {type(e).__name__}: {e}', None

with gr.Blocks() as demo:
    gr.Markdown('## StyleGAN3 Latent Space Viewer')
    with gr.Row():
        pkl_upload = gr.File(label='Upload .pkl checkpoint', type='filepath')
        load_btn = gr.Button('Load Model')
    status = gr.Textbox(label='Status', interactive=False)
    load_btn.click(load_model, inputs=pkl_upload, outputs=status)

    with gr.Tabs():
        with gr.Tab('Explore (manual)'):
            with gr.Row():
                seed_a = gr.Number(value=0, label='Seed A')
                seed_b = gr.Number(value=1, label='Seed B')
                random_btn = gr.Button('🎲 Randomize seeds')
            interp = gr.Slider(0, 1, value=0.0, step=0.01, label='Interpolate A → B')
            psi = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Truncation psi')
            gen_btn = gr.Button('Generate')
            output_img = gr.Image(label='Output')

            random_btn.click(randomize, outputs=[seed_a, seed_b])
            gen_btn.click(generate, inputs=[seed_a, seed_b, interp, psi], outputs=output_img)
            for control in [seed_a, seed_b, interp, psi]:
                control.change(generate, inputs=[seed_a, seed_b, interp, psi], outputs=output_img)

        with gr.Tab('Live Walk (auto)'):
            gr.Markdown(
                'Continuously wanders through latent space: picks a random target, '
                'interpolates toward it frame by frame, then picks a new target and keeps going — '
                'closest browser-based equivalent to the native visualizer\'s live traversal.'
            )
            with gr.Row():
                walk_psi = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Truncation psi')
                walk_fps = gr.Slider(1, 20, value=10, step=1, label='Target FPS')
                walk_steps = gr.Slider(10, 200, value=60, step=10, label='Steps per leg (higher = slower drift)')
            with gr.Row():
                start_btn = gr.Button('▶️ Start Walk')
                stop_btn = gr.Button('⏹️ Stop Walk')
            walk_img = gr.Image(label='Live Output')

            start_btn.click(start_walk, inputs=[walk_psi, walk_fps, walk_steps], outputs=walk_img)
            stop_btn.click(stop_walk)

        with gr.Tab('Feature Extraction (GANSpace)'):
            gr.Markdown(
                '**Purpose:** replicate the workshop Feature Extraction/GANSpace workflow. '
                'The video describes PCA as a way to find salient, interpretable directions '
                'in the loaded model\'s latent space. Feature 1 explains the most variance, '
                'then the later features generally explain less. The extracted directions '
                'belong to this model and should not be treated as interchangeable with another model.'
            )

            with gr.Row():
                feature_count = gr.Slider(
                    2, 50, value=10, step=1,
                    label='Number of features',
                    info='How many PCA directions to extract.'
                )
                sample_count = gr.Slider(
                    200, 5000, value=1000, step=100,
                    label='PCA samples',
                    info='Number of random W-space samples used to calculate PCA.'
                )
                feature_sparsity = gr.Slider(
                    0.1, 1.0, value=1.0, step=0.05,
                    label='Sparsity',
                    info='1.0 keeps the full PCA direction; lower values make the direction sparser.'
                )
                feature_seed = gr.Number(value=42, precision=0, label='Random seed')

            extract_btn = gr.Button('🔎 Get Salient Features', variant='primary')
            feature_status = gr.Textbox(label='Extraction result', lines=12, interactive=False)

            with gr.Row():
                save_feature_btn = gr.Button('💾 Save Feature Directions')
                feature_file = gr.File(label='Load saved .npz features', type='filepath')
                load_feature_btn = gr.Button('Load Features')

            load_feature_status = gr.Textbox(label='Feature file status', interactive=False)

            gr.Markdown('### Explore an extracted feature')
            with gr.Row():
                feature_index = gr.Dropdown(
                    choices=[],
                    value=None,
                    label='Feature / direction'
                )
                feature_strength = gr.Slider(
                    -4.0, 4.0, value=0.0, step=0.05,
                    label='Feature strength',
                    info='Negative and positive values move in opposite directions.'
                )
                feature_seed_image = gr.Number(value=0, precision=0, label='Image seed')
                feature_psi = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Truncation psi')
                layer_mode = gr.Dropdown(
                    ['All layers', 'Coarse layers', 'Fine layers'],
                    value='All layers',
                    label='Apply direction to'
                )

            feature_info_box = gr.Textbox(label='Selected feature', interactive=False)
            feature_generate_btn = gr.Button('Generate Feature Edit', variant='primary')

            with gr.Row():
                feature_base_img = gr.Image(label='Base image')
                feature_edit_img = gr.Image(label='Feature-edited image')

            extract_btn.click(
                extract_salient_features,
                inputs=[feature_count, sample_count, feature_sparsity, feature_seed],
                outputs=[feature_status, feature_index, feature_strength]
            )
            feature_index.change(feature_info, inputs=feature_index, outputs=feature_info_box)
            feature_generate_btn.click(
                apply_feature,
                inputs=[feature_seed_image, feature_psi, feature_index, feature_strength, layer_mode],
                outputs=[feature_base_img, feature_edit_img, feature_info_box]
            )

            save_feature_btn.click(save_features, outputs=feature_file)
            load_feature_btn.click(
                load_features,
                inputs=feature_file,
                outputs=[load_feature_status, feature_index]
            )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f48bd4d2f3ace8077f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting up PyTorch plugin "bias_act_plugin"... Done.
Setting up PyTorch plugin "filtered_lrelu_plugin"... Done.


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2344, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2102, in postprocess_data
    await processing_utils.async_move_files_to_cache(
    ...<3 lines>...
    )
  File "/usr/local/lib/python3.13/dist-package

Setting up PyTorch plugin "upfirdn2d_plugin"... Done.


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1702, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 924, in async_iteratio

### About Feature Extraction / GANSpace

This tab follows the **Feature Extraction** portion of the Autolume workshop. The workshop describes GANSpace as a PCA-based method for finding salient directions in the latent space, with earlier PCA components generally accounting for more variation than later ones.

The Colab implementation uses PCA over sampled **W-space** points from the loaded StyleGAN3 generator. It then lets you move a generated image in either direction along an extracted component. This is a notebook-friendly implementation of the same workflow rather than a byte-for-byte copy of Autolume's desktop feature-extractor internals.


### Troubleshooting
- **Custom op compile errors**: `Runtime -> Restart runtime`, then re-run cells from the top. Occasionally the CUDA toolkit version Colab ships changes and the first compile needs a clean environment.
- **Out of memory**: high-resolution models (1024px+) on a single T4 (16GB) should be fine for single-image generation; if you hit OOM, restart the runtime to clear GPU memory before reloading the model.
- **Slow first image**: expected — CUDA ops (`filtered_lrelu`, `bias_act`, `upfirdn2d`) JIT-compile on first use and are cached after that.